In [ ]:
import matplotlib.pyplot as plt
import spikeinterface as si
import spikeinterface.extractors as se
import spikeinterface.postprocessing as spost
import spikeinterface.widgets as sw
import numpy as np
from pathlib import Path
import shutil

In [ ]:
wave_a = np.load('C:\waveforms\waveforms/waveforms_1.npy')
wave_a_index = np.load('C:\waveforms\waveforms/sampled_index_1.npy')

In [ ]:
import pickle
import pandas as pd

# Load the pickle file
with open(r'S:/Sachuriga/Ephys_Vedio/CR_CA1/raw_files/63383_Open_Field_50Hz_A2024-07-13T14_10_11DLC_DlcrnetStride32Ms5_CR_implant_DLCnetNov30shuffle1_snapshot_350_sk.pickle', 'rb') as file:
    data = pickle.load(file)

# Check the type and contents
print("Type of data:", type(data))
print("Contents of data:", data)


In [ ]:
data[0][0][1][]

In [ ]:
data['scorer', 'individual1']

In [ ]:
for row in data[0]:
    

In [ ]:
dlc_path = Path("S:\Sachuriga\Ephys_Vedio\CR_CA1/65165_Open_Field_50Hz_A2023-06-29T17_03_02DLC_dlcrnetms5_CR_implant_DLCnetNov30shuffle3_600000_sk_filtered.h5")

In [ ]:
temp_vname = dlc_path.name.split("DLC_dlcrnet")
vname=temp_vname[0]
path_ori = dlc_path.parent
idun_vedio_path=r"P:/Overlap_project/data/CR_implant_add_new"

In [ ]:
cd Q:\sachuriga\Sachuriga_Python\quattrocolo-nwb4fp\src

In [ ]:
import sys
sys.path.append(r"Q:/sachuriga/Sachuriga_Python/quattrocolo-nwb4fp/src/")
from pickle import TRUE
import spikeinterface as si
import spikeinterface.extractors as se
import spikeinterface.postprocessing as post
from nwb4fp.postprocess.Get_positions import load_positions,load_positions_h5,test_positions_h5
from nwb4fp.postprocess.get_potential_merge import get_potential_merge
from spikeinterface.preprocessing import (bandpass_filter,
                                           common_reference,
                                           whiten)
import spikeinterface.exporters as sex
import spikeinterface.qualitymetrics as sqm
from pathlib import Path
import pandas as pd
from nwb4fp.postprocess.extract_wf import wf4unim,divide_wf
import spikeinterface.preprocessing as spre
import numpy as np

from spikeinterface.extractors.neoextractors.openephys import OpenEphysBinaryRecordingExtractor


In [ ]:
path=r"S:\Sachuriga/Ephys_Recording/CR_CA1/63383/63383_2024-07-20_14-21-42_A_phy_k"
global_job_kwargs = dict(n_jobs=32, total_memory="64G",mp_context= "spawn",progress_bar=True)
si.set_global_job_kwargs(**global_job_kwargs)
temp_folder = r"C:/Ephys_temp"

sorting = se.read_phy(folder_path=path, load_all_cluster_properties=True,exclude_cluster_groups = ["noise", "mua"])

temp_path = path.split("_phy")
raw_path = temp_path[0]
#stream_name = 'Record Node 101#OE_FPGA_Acquisition_Board-100.Rhythm Data'
stream_name  = OpenEphysBinaryRecordingExtractor(raw_path,stream_id='0').get_streams(raw_path)[0][0]
print(fr"Before mannual search the stream_name. Auto search result is {stream_name}")
try:
    recording = se.read_openephys(raw_path, stream_name=stream_name, load_sync_timestamps=True)
except AssertionError:
    try:
        stream_name = 'Record Node 102#OE_FPGA_Acquisition_Board-101.Rhythm Data'
        recording = se.read_openephys(raw_path, stream_name=stream_name, load_sync_timestamps=True)
    except AssertionError:
        stream_name = 'Record Node 101#Acquisition_Board-100.Rhythm Data'
        recording = se.read_openephys(raw_path, stream_name=stream_name, load_sync_timestamps=True)

import probeinterface as pi

# from probeinterface import plotting
manufacturer = 'cambridgeneurotech'
probe_name = 'ASSY-236-F'
probe = pi.get_probe(manufacturer, probe_name)
print(probe)
# probe.wiring_to_device('cambridgeneurotech_mini-amp-64')
# map channels to device indices
mapping_to_device = [
    # connector J2 TOP
    41, 39, 38, 37, 35, 34, 33, 32, 29, 30, 28, 26, 25, 24, 22, 20,
    46, 45, 44, 43, 42, 40, 36, 31, 27, 23, 21, 18, 19, 17, 16, 14,
    # connector J1 BOTTOM
    55, 53, 54, 52, 51, 50, 49, 48, 47, 15, 13, 12, 11, 9, 10, 8,
    63, 62, 61, 60, 59, 58, 57, 56, 7, 6, 5, 4, 3, 2, 1, 0
]

probe.set_device_channel_indices(mapping_to_device)
probe.to_dataframe(complete=True).loc[:, ["contact_ids", "shank_ids", "device_channel_indices"]]
probegroup = pi.ProbeGroup()
probegroup.add_probe(probe)

pi.write_prb(f"{probe_name}.prb", probegroup, group_mode="by_shank")
recording_prb = recording.set_probe(probe, group_mode="by_shank")
rec = bandpass_filter(recording_prb, freq_min=300, freq_max=8000)
bad_channel_ids, channel_labels = spre.detect_bad_channels(rec, method='coherence+psd',n_neighbors = 9)
#recording_good_ch= rec.remove_channels(bad_channel_ids)
recording_good_channels_f = spre.interpolate_bad_channels(rec,bad_channel_ids)

rec_save = common_reference(recording_good_channels_f, reference='global', operator='median')
rec_w = whiten(rec_save, int_scale=200, mode='local', radius_um=100.0)
                
sorting.set_property(key='group', values = sorting.get_property("channel_group"))
print(f"get times for raw sorts{sorting.get_times()}")
## step to analyzer
GLOBAL_KWARGS = dict(n_jobs=8, total_memory="64G", progress_bar=True, mp_context= "spawn", chunk_size=5000, chunk_duration="1s")
si.set_global_job_kwargs(**GLOBAL_KWARGS)

analyzer = si.create_sorting_analyzer(sorting=sorting, recording=rec_w, format='memory', folder=fr"{temp_folder}",overwrite=True)
## step to computations
GLOBAL_KWARGS = dict(n_jobs=12, total_memory="64G", progress_bar=True, mp_context= "spawn", chunk_size=5000, chunk_duration="1s")
si.set_global_job_kwargs(**GLOBAL_KWARGS)
we1 = analyzer.compute("random_spikes","waveforms")
we1 = analyzer.compute("waveforms")
we1 = analyzer.compute("noise_levels")
we1 = analyzer.compute("templates")
#get potential merging sorting objects
print("processing potential merge...\n")

In [ ]:
recording_good_channels_f.get_channel_ids()

In [ ]:
print(sorting.count_num_spikes_per_unit())

In [ ]:
sort_merge = get_potential_merge(sorting, analyzer)
print(sort_merge.count_num_spikes_per_unit())

In [ ]:
a=sorting.get_property_keys()
print(a)

In [ ]:
a = sorting.get_property("ch")
a